# Modele de detection de plaques — Ciment's EyeCe carnet entraine le modele `plate`, qui **localise** la plaque sur l'imaged'un vehicule. Il ne la lit pas : la lecture est faite ensuite par easyocr, surla zone que ce modele decoupe.Executez les cellules dans l'ordre. Tout est renseigne, il n'y a rien aremplir.## Pourquoi un modele dedieSans lui, la localisation se fait par traitement d'image classique (chapeauhaut-de-forme noir + Sobel). Cela repere des rectangles contrastes — dontbeaucoup ne sont pas des plaques : un autocollant, une calandre, un reflet.Chaque faux candidat coute une seconde d'OCR sur une machine a deux coeurs.Un modele YOLO dedie change deux choses : il ne propose que des plaques, et illes cadre serre, ce qui augmente nettement le taux de lecture.## Ce qui limite vraiment la lectureAvant d'entrainer quoi que ce soit, verifiez le cadrage. Mesure faite sur lavideo d'essai du projet : la plaque du vehicule detecte faisait **60 pixels delarge**. Aucun moteur ne lit cela. Il en faut **au moins 90, en pratique 120**.Aucun modele ne rattrapera une camera trop loin. Regardez d'abord`Parametres -> Etat du systeme -> Lecture des plaques` : la colonne « largeurvue » donne le chiffre reel de votre installation.

## 1. Environnement`Execution -> Modifier le type d'execution -> GPU T4`. Verifiez que la cellulesuivante affiche bien une carte : sans GPU, l'entrainement prendrait desdizaines d'heures.

In [ ]:
!nvidia-smi!pip install -q ultralytics roboflow

## 2. Le jeu de donnees**License Plate Recognition** (Roboflow Universe), une seule classe :`License_Plate`.Deux versions utilisables, et le choix depend surtout de votre patience :| version | images | entrainement | duree sur T4 ||---|---|---|---|| **11** | 10 125 | 7 057 | environ 45 min || 4 | 24 242 | 21 174 | environ 2 h 30 |La **11** est prise par defaut. Sept mille plaques suffisent largement pour uneclasse unique, et une session Colab gratuite se coupe souvent avant deuxheures et demie. Passez a la 4 si vous voulez un modele plus robuste et quevous pouvez laisser tourner.> La cle ci-dessous est celle de votre espace Roboflow. Elle est visible dans> le depot GitHub, qui est public : **revoquez-la apres l'entrainement**> (Roboflow -> Account -> Roboflow Keys -> Revoke), puis regenerez-en une.

In [ ]:
from roboflow import RoboflowCLE = "kQle1ihpBmsqoXROy27k"VERSION = 11          # mettez 4 pour le jeu complet (24 242 images)rf = Roboflow(api_key=CLE)projet = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")jeu = projet.version(VERSION).download("yolov8")print("jeu telecharge dans :", jeu.location)

## 3. Verifier ce qu'on a telechargeUne minute ici evite une heure d'entrainement sur un jeu mal decompresse.

In [ ]:
import yamlfrom pathlib import Pathconfig = yaml.safe_load(open(f"{jeu.location}/data.yaml"))print("classes :", config["names"])for partie in ("train", "valid", "test"):    dossier = Path(jeu.location) / partie / "images"    if dossier.exists():        print(f"{partie:<6} {len(list(dossier.glob('*')))} images")

## 4. Entrainement`imgsz=640` est **obligatoire** : c'est la taille figee a l'export OpenVINO duprojet, et la changer casse l'inference.`yolov8n` suffit — une plaque est un objet simple et tres regulier, et lemodele tournera sur un processeur a deux coeurs, ou le nano est le seulraisonnable.

In [ ]:
from ultralytics import YOLOmodele = YOLO("yolov8n.pt")modele.train(    data=f"{jeu.location}/data.yaml",    epochs=40,    imgsz=640,    batch=32,    patience=10,    name="plaques",)

## 5. Verifier avant de livrerRegardez le **mAP50**. En dessous de 0,85, le modele cadrera mal et la lectureen patira — relancez avec plus d'epoques ou passez a la version 4 du jeu.

In [ ]:
metriques = modele.val()print("mAP50   :", round(float(metriques.box.map50), 3))print("mAP50-95:", round(float(metriques.box.map), 3))

## 6. Essayer sur une vraie image de votre portailDeposez une photo prise par la camera du site (icone dossier a gauche), puisadaptez le nom ci-dessous. C'est le seul essai qui compte : le mAP mesure laperformance sur le jeu de donnees, pas sur votre installation.

In [ ]:
# resultats = modele.predict("portail.jpg", conf=0.35, save=True)# from IPython.display import Image# Image("runs/detect/predict/portail.jpg", width=700)

## 7. Recuperer le modeleTelechargez `best.pt`, renommez-le **`ciments_eye_plate_best.pt`** et deposez-ledans le dossier `models/` du projet. Puis, sur la machine du site :```python scripts/export_openvino.py```Ajoutez enfin cette entree dans `config/config.yaml`, section `models` :```yaml  plate:    file: models/ciments_eye_plate_best_openvino_model    conf: 0.35    enabled: true```Le lecteur de plaques s'en sert automatiquement des qu'il est declare — il n'ya rien d'autre a changer dans le code.

In [ ]:
from google.colab import filesfiles.download("runs/detect/plaques/weights/best.pt")